In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,random_split,Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import random
import os

In [ ]:
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
import os
from collections import defaultdict

def load_cedar_dataset(root_path):

    org_path = os.path.join(root_path, "/content/drive/MyDrive/signatures/full_org")
    forg_path = os.path.join(root_path, "/content/drive/MyDrive/signatures/full_forg")

    data_dict = defaultdict(lambda: {"org": [], "forg": []})

    # Load Genuine Signatures
    for filename in os.listdir(org_path):
        if filename.endswith(".png"):

            # example: original_1_1.png
            writer_id = int(filename.split("_")[1])

            full_path = os.path.join(org_path, filename)
            data_dict[writer_id]["org"].append(full_path)

    # Load Forged Signatures
    for filename in os.listdir(forg_path):
        if filename.endswith(".png"):

            # example: forgeries_1_1.png
            writer_id = int(filename.split("_")[1])

            full_path = os.path.join(forg_path, filename)
            data_dict[writer_id]["forg"].append(full_path)

    return data_dict

In [ ]:
root_path = "/content/drive/MyDrive/signatures"
data_dict = load_cedar_dataset(root_path)

writers = list(data_dict.keys())

print("Total Writers:", len(writers))
print("Writer 1 Genuine:", len(data_dict[1]["org"]))
print("Writer 1 Forged:", len(data_dict[1]["forg"]))

Total Writers: 55
Writer 1 Genuine: 24
Writer 1 Forged: 24


In [ ]:
random.shuffle(writers)

split = int(0.8 * len(writers))
train_writers = writers[:split]
test_writers  = writers[split:]

In [ ]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))   # fixes size automatically
        )

        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward_once(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        return x

    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        return out1, out2

In [ ]:
from PIL import Image
class SignaturePairDataset(Dataset):
    def __init__(self, data_dict, writers, transform=None):
        self.data_dict = data_dict
        self.writers = writers
        self.transform = transform

    def __len__(self):
        return 10000   # you control number of pairs per epoch

    def __getitem__(self, index):

        # 50% positive, 50% negative
        if random.random() < 0.5:
            # Positive pair
            writer = random.choice(self.writers)
            img1, img2 = random.sample(self.data_dict[writer]["org"], 2)
            label = 1
        else:
            writer = random.choice(self.writers)

            if random.random() < 0.5:
                img1 = random.choice(self.data_dict[writer]["org"])
                img2 = random.choice(self.data_dict[writer]["forg"])
            else:
                writer2 = random.choice(self.writers)
                img1 = random.choice(self.data_dict[writer]["org"])
                img2 = random.choice(self.data_dict[writer2]["org"])

            label = 0

        img1 = Image.open(img1).convert("L")
        img2 = Image.open(img2).convert("L")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [ ]:
train_dataset = SignaturePairDataset(
    data_dict=data_dict,
    writers=train_writers,
    transform=transform
)

test_dataset = SignaturePairDataset(
    data_dict=data_dict,
    writers=test_writers,
    transform=transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        distance = F.pairwise_distance(output1, output2)

        loss = torch.mean(
            label * torch.pow(distance, 2) +
            (1 - label) * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)
        )

        return loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


In [ ]:
for epoch in range(10):
    model.train()
    running_loss = 0

    for img1, img2, label in train_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)

        optimizer.zero_grad()
        out1, out2 = model(img1, img2)

        loss = criterion(out1, out2, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.33960625853020543
Epoch 2, Loss: 0.23062871567928753
Epoch 3, Loss: 0.19931711696873838
Epoch 4, Loss: 0.16956215032849448
Epoch 5, Loss: 0.15893991026110924
Epoch 6, Loss: 0.14461134991849572
Epoch 7, Loss: 0.12910897576342376
Epoch 8, Loss: 0.12483053828199832
Epoch 9, Loss: 0.1154541563325987
Epoch 10, Loss: 0.11323228047583431


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

distances = []
labels = []

model.eval()
with torch.no_grad():
    for img1, img2, label in test_loader:
        out1, out2 = model(img1.to(device), img2.to(device))
        distance = F.pairwise_distance(out1, out2)
        distances.extend(distance.cpu().numpy())
        labels.extend(label.numpy())

fpr, tpr, thresholds = roc_curve(labels, distances, pos_label=1)

fnr = 1 - tpr
eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]

print("EER:", eer)

EER: 0.9211734184793454


In [ ]:
threshold = 1.0   # you can tune this

correct = 0
total = 0

model.eval()

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2 = img1.to(device), img2.to(device)

        out1, out2 = model(img1, img2)
        distances = F.pairwise_distance(out1, out2)

        for d, y in zip(distances, label):
            pred = 1 if d.item() < threshold else 0

            if pred == y.item():
                correct += 1
            total += 1

accuracy = correct / total

print("Total Correct:", correct)
print("Total Samples:", total)
print("Accuracy:", accuracy)

Total Correct: 9237
Total Samples: 10000
Accuracy: 0.9237


In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms

# ----------------------------
# 1. DEVICE
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 2. LOAD MODEL
# ----------------------------
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()

# ----------------------------
# 3. TRANSFORM (must match training)
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ----------------------------
# 4. LOAD IMAGE FUNCTION
# ----------------------------
def load_image(path):
    img = Image.open(path).convert("L")
    img = transform(img).unsqueeze(0).to(device)
    return img

# ----------------------------
# 5. SINGLE PAIR TEST FUNCTION
# ----------------------------
def test_pair(img1_path, img2_path, threshold=1.0):

    img1 = load_image(img1_path)
    img2 = load_image(img2_path)

    with torch.no_grad():
        out1, out2 = model(img1, img2)
        distance = F.pairwise_distance(out1, out2).item()

    print("Image 1:", img1_path)
    print("Image 2:", img2_path)
    print("Distance:", round(distance, 4))

    if distance < threshold:
        print("Result: ✔ MATCH (Same Writer)")
    else:
        print("Result: ❌ NOT MATCH (Forged/Different Writer)")

    print("-" * 50)

    return distance

# ----------------------------
# 6. MULTI-TEST FUNCTION (FIXED)
# ----------------------------
def test_against_many(img1_path, img2_list, threshold=1.0):

    for img2_path in img2_list:
        test_pair(img1_path, img2_path, threshold)

# ----------------------------
# 7. EXAMPLE USAGE
# ----------------------------

img1_path = "/content/drive/MyDrive/signatures/full_org/original_1_1.png"

img2_paths = [
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_1.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_2.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_3.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_4.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_5.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_6.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_7.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_8.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_9.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_10.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_11.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_12.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_13.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_14.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_15.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_16.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_17.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_18.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_19.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_20.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_21.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_22.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_23.png",
    "/content/drive/MyDrive/signatures/full_org/original_1_24.png"
]

test_against_many(img1_path, img2_paths, threshold=1.0)

Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_1.png
Distance: 4.5213
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_2.png
Distance: 4.6056
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_3.png
Distance: 4.6896
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_4.png
Distance: 4.6462
Result: ❌ NOT MATCH (Forged/Different Writer)
-----------------------------------

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms

# ----------------------------
# 1. DEVICE
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 2. LOAD MODEL
# ----------------------------
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()

# ----------------------------
# 3. TRANSFORM (must match training)
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ----------------------------
# 4. LOAD IMAGE FUNCTION
# ----------------------------
def load_image(path):
    img = Image.open(path).convert("L")
    img = transform(img).unsqueeze(0).to(device)
    return img

# ----------------------------
# 5. SINGLE PAIR TEST FUNCTION
# ----------------------------
def test_pair(img1_path, img2_path, threshold=1.0):

    img1 = load_image(img1_path)
    img2 = load_image(img2_path)

    with torch.no_grad():
        out1, out2 = model(img1, img2)
        distance = F.pairwise_distance(out1, out2).item()

    print("Image 1:", img1_path)
    print("Image 2:", img2_path)
    print("Distance:", round(distance, 4))

    if distance < threshold:
        print("Result: ✔ MATCH (Same Writer)")
    else:
        print("Result: ❌ NOT MATCH (Forged/Different Writer)")

    print("-" * 50)

    return distance

# ----------------------------
# 6. MULTI-TEST FUNCTION (FIXED)
# ----------------------------
def test_against_many(img1_path, img2_list, threshold=1.0):

    for img2_path in img2_list:
        test_pair(img1_path, img2_path, threshold)

# ----------------------------
# 7. EXAMPLE USAGE
# ----------------------------

img1_path = "/content/drive/MyDrive/signatures/full_org/original_1_1.png"

img2_paths = [
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_1.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_2.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_3.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_4.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_5.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_6.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_7.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_8.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_9.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_10.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_11.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_12.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_13.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_14.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_15.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_16.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_17.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_18.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_19.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_20.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_21.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_22.png",
    "/content/drive/MyDrive/signatures/full_forg/forgeries_1_23.png",
    "/content/drive/MyDrive/signatures/full_org/original_1_24.png"
]

test_against_many(img1_path, img2_paths, threshold=1.0)

Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_1.png
Distance: 4.5213
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_2.png
Distance: 4.6056
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_3.png
Distance: 4.6896
Result: ❌ NOT MATCH (Forged/Different Writer)
--------------------------------------------------
Image 1: /content/drive/MyDrive/signatures/full_org/original_1_1.png
Image 2: /content/drive/MyDrive/signatures/full_forg/forgeries_1_4.png
Distance: 4.6462
Result: ❌ NOT MATCH (Forged/Different Writer)
-----------------------------------